In [1]:
from google.colab import drive
drive.mount ('/content/drive')

Mounted at /content/drive


In [2]:
!ls

drive  sample_data


In [3]:
%cd /content/drive/MyDrive/ENGG680_2025_Fall/ENGG680_Project_G07

/content/drive/MyDrive/ENGG680_2025_Fall/ENGG680_Project_G07


In [4]:
#ENGG680-LO1-F25-G7 Ground Improvement ML – Random Forest Regression Workflow

# This notebook contains the following:
# - Reads an existing CSV dataset
# - 21 input features (incl. stone-column features) → 6 output targets
# - Handles missing data:
#   • Drop rows with >2 missing feature values
#   • Impute remaining missing values with regional (source-wise) medians
# - Removes outliers: mean ± 3σ (~0.5% of samples)
# - 70/15/15 train/val/test split
# - Random Forest Regressor (multi-output)
# - Feature importance + plots
# - Live prediction helper function + field validation profiles

In [7]:
# 1. Library Imports

import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

import matplotlib.pyplot as plt

np.random.seed(42)

In [8]:
# 2. Load Existing Dataset
#
# Change the filename below if your CSV is named differently.

# %%
csv_path = "synthetic_ground_improvement_dataset.csv"  # csv containing the dataset
df_all = pd.read_csv(csv_path)

print("Raw dataset shape:", df_all.shape)
print("Columns:")
print(df_all.columns.tolist())

df_all.head()

Raw dataset shape: (1100, 28)
Columns:
['source', 'sand_pct', 'silt_pct', 'clay_pct', 'LL', 'PI', 'moisture_%', 'gamma_kN_m3', 'e_void_ratio', 'c_kPa', 'phi_deg', 'depth_m', 'OCR', 'sigma_v_eff_kPa', 'E_MPa', 'stone_column_flag', 'relative_density_Dr', 'stone_col_D_m', 'stone_col_S_m', 'area_replacement_ratio_Ar', 'Ec_MPa', 'Ec_over_Es', 'delta_c_kPa', 'delta_phi_deg', 'delta_E_MPa', 'delta_e', 'delta_gamma_kN_m3', 'effectiveness_%']


,source,sand_pct,silt_pct,clay_pct,LL,PI,moisture_%,gamma_kN_m3,e_void_ratio,c_kPa,...,stone_col_S_m,area_replacement_ratio_Ar,Ec_MPa,Ec_over_Es,delta_c_kPa,delta_phi_deg,delta_E_MPa,delta_e,delta_gamma_kN_m3,effectiveness_%
0,CANSIS,5.898700,68.501593,25.599707,55.361932,23.956580,14.796683,16.808528,0.687332,28.777853,...,0.0,0.0,0.0,0.0,26.930557,5.633408,58.303891,-0.209706,1.471310,49.864218
1,CANSIS,37.325397,5.385316,57.289287,27.626952,5.614673,18.084289,16.734894,0.539442,32.236964,...,0.0,0.0,0.0,0.0,31.210854,5.889247,71.918773,-0.221278,1.368215,53.285867
2,CANSIS,23.933889,30.806444,45.259667,48.814418,5.553089,15.666410,20.031273,0.725296,33.311048,...,0.0,0.0,0.0,0.0,29.280265,4.999389,53.842052,-0.199178,1.169152,39.931544
3,CANSIS,42.179820,19.893963,37.926217,50.335752,13.090255,18.874908,16.747731,0.671170,33.804068,...,0.0,0.0,0.0,0.0,30.929047,5.345075,60.103286,-0.220157,1.377524,55.219563
4,CANSIS,20.230171,66.188804,13.581025,41.206534,17.216080,11.735596,16.859749,0.872416,20.643789,...,0.0,0.0,0.0,0.0,18.701479,4.264439,50.984805,-0.192065,1.252835,43.864908


In [9]:
# 3. Define Feature and Target Columns

# These names must match the columns in your CSV/excel file.
# If your file uses slightly different names, update the lists below.

feature_cols = [
    "sand_pct",
    "silt_pct",
    "clay_pct",
    "LL",
    "PI",
    "moisture_%",
    "gamma_kN_m3",
    "e_void_ratio",
    "c_kPa",
    "phi_deg",
    "depth_m",
    "OCR",
    "sigma_v_eff_kPa",
    "E_MPa",
    # stone-column features
    "stone_column_flag",
    "relative_density_Dr",
    "stone_col_D_m",
    "stone_col_S_m",
    "area_replacement_ratio_Ar",
    "Ec_MPa",
    "Ec_over_Es",
]

target_cols = [
    "delta_c_kPa",
    "delta_phi_deg",
    "delta_E_MPa",
    "delta_e",
    "delta_gamma_kN_m3",
    "effectiveness_%"
]

# For data checking: ensure all columns exist
missing_features = [c for c in feature_cols if c not in df_all.columns]
missing_targets = [c for c in target_cols if c not in df_all.columns]

if missing_features or missing_targets:
    raise ValueError(
        f"Missing features: {missing_features}\nMissing targets: {missing_targets}"
    )

print("All required feature/target columns found.")

All required feature/target columns found.


In [11]:
# 4. Handle Missing Values

# - Drop rows with **more than 2 missing feature values** (too incomplete).
# - For remaining rows, input missing feature values using the
#   **median within each data source** (`source`: CANSIS / Provincial / Commercial / Stone-Column).
# - If a feature is still missing after that, fall back to the **global median**.

# Count missing values per row across FEATURES only
row_missing_counts = df_all[feature_cols].isna().sum(axis=1)

before_rows = df_all.shape[0]
df_all = df_all.loc[row_missing_counts <= 2].copy()
after_rows = df_all.shape[0]

print(f"Dropped {before_rows - after_rows} rows with >2 missing feature values.")
print("Shape after row drop:", df_all.shape)

# Use 'source' column as regional grouping if available
if "source" in df_all.columns:
    group_col = "source"
    print(f"Inputting missing values using regional medians by '{group_col}'.")
else:
    group_col = None
    print("WARNING: 'source' column not found; inputting with global medians only.")

# Input missing feature values
for col in feature_cols:
    if df_all[col].isna().any():
        if group_col is not None:
            # Fill using median within each region (source)
            df_all[col] = (
                df_all.groupby(group_col)[col]
                .transform(lambda s: s.fillna(s.median()))
            )
        # Fallback: global median
        if df_all[col].isna().any():
            global_median = df_all[col].median()
            df_all[col].fillna(global_median, inplace=True)

total_missing_after = df_all[feature_cols].isna().sum().sum()
print(f"Total remaining missing feature values: {total_missing_after}")


# 5.  Remove Outliers (|x - mean| > 3σ)
# - Applied only to input features (not targets)
# - Removes extreme outliers (~0.5% of samples)
# - Improves stability of Random Forest (RFR) predictions

before_outliers = df_all.shape[0]

feature_data = df_all[feature_cols]
feature_means = feature_data.mean()
feature_stds  = feature_data.std()

# Boolean mask: rows where all features are within mean ± 3σ
within_3sigma = ((feature_data - feature_means).abs() <= 3 * feature_stds).all(axis=1)

df_all = df_all[within_3sigma].copy()

after_outliers = df_all.shape[0]
removed = before_outliers - after_outliers

print(f"Removed {removed} outlier rows using mean ± 3σ rule.")
print(f"Dataset shape after outlier removal: {df_all.shape}")

Dropped 0 rows with >2 missing feature values.
Shape after row drop: (1078, 28)
Inputting missing values using regional medians by 'source'.
Total remaining missing feature values: 0
Removed 12 outlier rows using mean ± 3σ rule.
Dataset shape after outlier removal: (1066, 28)


In [12]:
# 6. Train / Validation / Test Split (70 / 15 / 15)

X = df_all[feature_cols].values
y = df_all[target_cols].values

# First split: Train (70%) vs Temp (30%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)

# Second split: Val (15%) vs Test (15%) – 50/50 of the 30%
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print("Train size:", X_train.shape[0])
print("Val size  :", X_val.shape[0])
print("Test size :", X_test.shape[0])


# 7. Random Forest Regressor Model (Training + Cross-Validation)

rf = RandomForestRegressor(
    n_estimators=600,
    max_depth=None,
    max_features=0.6,
    min_samples_split=2,
    min_samples_leaf=1,
    bootstrap=True,
    n_jobs=-1,
    random_state=42,
)

# 5-fold CV on training set
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf, X_train, y_train, cv=kf, scoring="r2")

print(f"5-fold CV R²: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

# Fit on train + validation
rf.fit(
    np.vstack([X_train, X_val]),
    np.vstack([y_train, y_val])
)

Train size: 746
Val size  : 160
Test size : 160
5-fold CV R²: 0.728 ± 0.024


RandomForestRegressor(max_features=0.6, n_estimators=600, n_jobs=-1,
                      random_state=42)